# Final Project: Network Door Security System - PYNQ #2

Device Roles
PYNQ #1: Controls the alarm function of the door security system. Will emit a loud buzzing sound and flash a RGB board bright red when sound is detected with a sound sensor. This board will automatically be listening for the sound sensor board when the code is run.

PYNQ #2: Controls the sound sensor board. When a sound is detected it will send a signal to PYNQ board #2 and buzz the buzzer and flash the LED.

Wiring to this board:

Sensor Module Wiring to PYNQ PMODA
- A0 pin unconnected 
- (+) pin connected to 3.3V
- (-) pin connected to GND
- D0 pin connected to Pin 1

In [3]:
from multiprocessing import Process
import threading
import time
from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")
import socket
import sys
import os

btns = base.btns_gpio
button_pressed = True

In [4]:
%%microblaze base.PMODA
#include "gpio.h"

static int inited = 0;
static gpio sound_pin;

void init_sound_sensor()
{
    if (inited) return;

    // PMODA pin 1
    sound_pin = gpio_open(1);          
    gpio_set_direction(sound_pin, GPIO_IN);

    inited = 1;
}

// Return current digital value (0 or 1)
unsigned int read_sound()
{
    if (!inited) init_sound_sensor();
    return gpio_read(sound_pin);
}

In [3]:
# Sound sensor board test code

# default state of the sound sensor, no noise detected
last_state = 0

# always listening while loop
while True:
    # the current state of sound sensor being read when reading PMODA PIN1
    current_state = read_sound()

    # rapid change in value between 0 and 1 means sound is detected
    if current_state == 1 and last_state == 0:
        print("Sound detected!")
        
    last_state = current_state
    # 0.01 second delay
    time.sleep(0.01)   

Sound detected!



KeyboardInterrupt



In [118]:
def client(button_pressed):

    # Enter the other PYNQ board's IP Address
    # HOST = '192.168.XXX.XXX'
    # HOST = '192.168.0.204'
    # HOST = '192.168.230.2'
    HOST = '192.168.0.44'   
    
    # Loopback to IP Address of my PYNQ Board
    # HOST = '127.0.0.1'
    PORT = 50007
    
    # creating a socket
    s_client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    
    # 1: Bind the socket to the pynq board <CLIENT-IP> at port <LISTENING-PORT>
    s_client.connect((HOST, PORT))
    # sets the socket in non-blocking mode, so that buttons work bi-directionally
    s_client.setblocking(False)
    print("Client connected to server")
    
    # The board starts disarmed
    armed = False
    print("\nBUTTON OPTIONS:")
    print("Press Button 1 to arm security system...")
    print("Press Button 3 to disconnect...")
    
    #with conn:
    print("\n------------------------------------------------")
    # print('Connected by', addr)

    while True:

        # Button 1: Case where the security system is armed with Button 1 on PYNQ2
        if base.buttons[1].read() == 1:
            print("\n------------------------------------------------")
            print("Button 1 Pressed - Security system armed")
            print("Listening for intruders...")
            
            print("\nBUTTON OPTIONS:")
            print("Press Button 2 to disarm security system...")
            print("Press Button 3 to disconnect from server...")
            
            # arms sound sensor
            armed = True
            # Sends sound sensor arming message
            s_client.sendall(b'ARM')
            time.sleep(0.3)

        # Button 2 Case: Disarm when the alarm board is actively buzzing/flashing with Button 2 on PYNQ2
        if base.buttons[2].read() == 1:
            print("\n------------------------------------------------")
            print("Button 2 Pressed - Security system disarmed")
            
            print("\nBUTTON OPTIONS:")
            print("Press Button 1 to arm security system...")
            print("Press Button 3 to disconnect from server...")
            
            # turns the alarm off
            alarm_active = False
            # disarms sound sensor
            armed = False
            # Sends sound sensor disarming message
            s_client.sendall(b'DISARM')
            time.sleep(0.3)

        # Button 3 Case: Disconnect when the alarm board is actively buzzing/flashing with Button 3 on PYNQ2
        if base.buttons[3].read() == 1:
            print("\n------------------------------------------------")
            print("Button 3 Pressed - Sound sensor board disconnected from server")
            
            # Sends sound sensor disconnect message
            s_client.sendall(b'DISCONNECT')
            # breaks out of loop is disconnected
            break

        try:
            # received button data at full bytes from PYNQ #1 Board
            pynq1_data = s_client.recv(1024)

            # Button 1: Case where the security system is armed with Button 1 on PYNQ1
            if pynq1_data == b'ARM':
                print("\n------------------------------------------------")
                print("Button 1 Pressed - Security system armed")
                print("Listening for intruders...")

                print("\nBUTTON OPTIONS:")
                print("Press Button 2 to disarm security system...")
                print("Press Button 3 to disconnect from server...")
                # arms sound sensor
                armed = True

            # Button 2 Case: Disarm when the alarm board is actively buzzing/flashing with Button 2 on PYNQ1
            elif pynq1_data == b'DISARM':
                print("\n------------------------------------------------")
                print("Button 2 Pressed - Security system disarmed")

                print("\nBUTTON OPTIONS:")
                print("Press Button 1 to arm security system...")
                print("Press Button 3 to disconnect from server...")
                
                # turns the alarm off
                alarm_active = False
                # disarms sound sensor
                armed = False

            # Alarm running case: alarm will run if alarm message is received AND the sound sensor is armed on PYNQ1
            elif pynq1_data == b'ALARM':
                if armed:
                    print("\n------------------------------------------------")
                    print("UNAUTHORIZED ACCESS DETECTED")

                    print("\nBUTTON OPTIONS:")
                    print("Press Button 2 to disarm security system...")
                    print("Press Button 3 to disconnect from server...")
                    
                    # turns the alarm on
                    alarm_active = True

            # Button 3 Case: Disconnect when the alarm board is actively buzzing/flashing with Button 3 on PYNQ1
            elif pynq1_data == b'DISCONNECT':
                print("\n------------------------------------------------")
                print("Button 3 Pressed - Alarm board disconnected from client")
                # Break out of loop if button 3 is pressed
                break

        # ignores potential blocks 
        except BlockingIOError:
            pass
        
        # Sound sensor code
        if armed:
            # reads sound sensor board digital pin
            sound = read_sound()
            # if sound is detected will send alarm message to PYNQ1
            if sound == 1:
                print("\n------------------------------------------------")
                print("UNAUTHORIZED ACCESS DETECTED")
                
                print("\nBUTTON OPTIONS:")
                print("Press Button 2 to disarm security system...")
                print("Press Button 3 to disconnect from server...")
                # Sound sensor detects an intruder. Send message to PYNQ1
                s_client.sendall(b'ALARM')
                time.sleep(1)
            
            time.sleep(0.1)
         
    # closes socket after the loop
    s_client.close()
    print("Client socket closed")

In [119]:
# Server process turns on upon code execution

try:

    # Button 0 Starts the Client
    print("Press Button 0 to connect to alarm board...")
    while base.buttons[0].read() == 0:
        time.sleep(0.1)

    print("\n------------------------------------------------")
    print("Button 0 Pressed")
    print("starting client")

    # Client process definition
    p1 = Process(target=client, args=(True,))
    # Starts client process
    p1.start()

    # Completes client process
    p1.join()
    print("Cell execution complete")
    
# Kill processes if button doesnt work (For debugging)
except KeyboardInterrupt:
    p.terminate()
    # print("Server process terminated")
    p.join()
    print("Cell execution complete")
    print('Interrupt')
    sys.exit(0)

Press Button 0 to connect to alarm board...

------------------------------------------------
Button 0 Pressed
starting client
Client connected to server

BUTTON OPTIONS:
Press Button 1 to arm security system...
Press Button 3 to disconnect...

------------------------------------------------

------------------------------------------------
Button 1 Pressed - Security system armed
Listening for intruders...

BUTTON OPTIONS:
Press Button 2 to disarm security system...
Press Button 3 to disconnect from server...

------------------------------------------------
Button 1 Pressed - Security system armed
Listening for intruders...

BUTTON OPTIONS:
Press Button 2 to disarm security system...
Press Button 3 to disconnect from server...

------------------------------------------------
UNAUTHORIZED ACCESS DETECTED
BUTTON OPTIONS:

Press Button 2 to disarm security system...
Press Button 3 to disconnect from server...

------------------------------------------------
Button 2 Pressed - Securit